# Data Leakage in Machine Learning

**Data Leakage** occurs when information from outside the training dataset is inadvertently used to build the model. This results in overly optimistic performance estimates during training and validation but leads to poor generalization on new, unseen data.

Rather than just describing leakage, this note makes it happen **on purpose** — three times — so you can see exactly how much each leak inflates your metrics, and how little it costs to prevent them. Every number below is produced by actually running this notebook.

## 1. What is Data Leakage?

- **Definition:** Data leakage happens when data that would not be available at prediction time is used during the model training process. This can be due to improperly partitioned data, inclusion of future information, or features that contain direct or indirect information about the target variable.

- **Causes of Data Leakage:**
  - **Temporal Leakage:** Using future data (e.g., including data that occurs after the event you're trying to predict).
  - **Target Leakage:** Including features that are highly correlated with the target because they were generated using the target variable or are proxies for it.
  - **Improper Data Splitting:** Overlapping data between the training and test sets, such as duplicate entries or shared information that should be exclusive to one set.

- **Consequences:**
  - **Overfitting:** The model learns patterns that are not generalizable.
  - **Inflated Metrics:** Performance measures (accuracy, AUC, etc.) are unrealistically high during cross-validation or testing.
  - **Poor Real-World Performance:** When deployed, the model performs significantly worse because it doesn't have access to the leaked information.

## 2. See It Happen — Three Experiments

### Experiment 1 — A feature that knows the answer (target leakage)

We build a synthetic dataset (2,000 rows, 10 features, 30% positives) and then add one extra feature, `post_outcome_score`, computed as **the target itself plus noise**. This mimics real-world features like *"number of customer-service calls recorded after the churn decision was confirmed"* — a value that only exists because the outcome already happened.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

X, y = make_classification(
    n_samples=2000, n_features=10, n_informative=5,
    weights=[0.7, 0.3], random_state=42
)

rng = np.random.default_rng(42)
leaky_feature = y + rng.normal(0, 0.3, size=len(y))   # target + noise

X_leaky = np.column_stack([X, leaky_feature])          # with the leak
X_honest = X                                           # without it

model = LogisticRegression(max_iter=1000)
auc_leaky  = cross_val_score(model, X_leaky,  y, cv=5, scoring='roc_auc')
auc_honest = cross_val_score(model, X_honest, y, cv=5, scoring='roc_auc')

print('With the leaked feature : AUC per fold', np.round(auc_leaky, 3),
      '| mean', round(auc_leaky.mean(), 3))
print('Without the leaked feat : AUC per fold', np.round(auc_honest, 3),
      '| mean', round(auc_honest.mean(), 3))

With the leaked feature : AUC per fold [0.996 0.994 0.989 0.991 0.993] | mean 0.993
Without the leaked feat : AUC per fold [0.874 0.868 0.851 0.809 0.791] | mean 0.839


**What just happened.** The leaky feature pushed cross-validated AUC from **0.839** to **0.993** — a 15-point jump that is *pure illusion*. Every fold of the leaky model looks production-ready (0.989–0.996), and no amount of cross-validation catches it, because **the leak is inside every fold**. The only fix is to not have the feature: if a value only exists after the outcome, it must never enter the feature matrix.

### Experiment 2 — Feature selection outside the pipeline

The classic study-case (Cawley & Talbot, 2010): 300 rows, 500 features, of which only 5 are informative. We want to select the best 10 features. The wrong way peeks at **all rows** (including future test rows) when scoring features; the right way wraps selection inside a `Pipeline`, so each cross-validation fold sees only its own training data.

In [2]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline import make_pipeline

X2, y2 = make_classification(
    n_samples=300, n_features=500, n_informative=5, random_state=42
)
lr = LogisticRegression(max_iter=2000)

# WRONG: score every feature using ALL rows, then "cross-validate"
selector = SelectKBest(mutual_info_classif, k=10).fit(X2, y2)
wrong = cross_val_score(lr, selector.transform(X2), y2, cv=5)

# RIGHT: selection lives inside the pipeline, re-fit on each fold's train part
pipe = make_pipeline(SelectKBest(mutual_info_classif, k=10), lr)
right = cross_val_score(pipe, X2, y2, cv=5)

print(f'WRONG (select before CV) : {wrong.mean():.3f} +- {wrong.std():.3f}')
print(f'RIGHT (select in CV)     : {right.mean():.3f} +- {right.std():.3f}')
print(f'Inflated by {100 * (wrong.mean() - right.mean()):.1f} accuracy points')

WRONG (select before CV) : 0.827 +- 0.058
RIGHT (select in CV)     : 0.707 +- 0.119
Inflated by 12.0 accuracy points


**What just happened.** The wrong workflow reported **0.827** while the honest estimate was **0.707** — the selection step leaked the test rows' label information into the feature ranking, worth a full **12 accuracy points** of optimism. Notice the second lie too: the leaky result also has a *lower* standard deviation (±0.058 vs ±0.119), so it doesn't just look better — it looks *more reliable*. On a real project you would have shipped a model 12 points worse than promised.

### Experiment 3 — The invisible leak (preprocessing before the split)

Fitting a `StandardScaler` on the full dataset before splitting is the most common "small" leak. Let's measure it honestly: KNN (k=7) on standardized data, 5-fold cross-validation repeated 50 times — scaling before the split vs. scaling inside a `Pipeline`.

In [3]:
from sklearn.model_selection import RepeatedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)
knn = KNeighborsClassifier(n_neighbors=7)

# WRONG: scaler learns means/stds from ALL rows first
X_scaled = StandardScaler().fit_transform(X)
wrong = cross_val_score(knn, X_scaled, y, cv=rkf, scoring='roc_auc')

# RIGHT: the pipeline re-fits the scaler inside each fold's training part
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
right = cross_val_score(pipe, X, y, cv=rkf, scoring='roc_auc')

print(f'WRONG (scale before split): {wrong.mean():.4f} +- {wrong.std():.4f}')
print(f'RIGHT  (scale in pipeline) : {right.mean():.4f} +- {right.std():.4f}')
print(f'Gap: {100 * (wrong.mean() - right.mean()):.2f} AUC points')

WRONG (scale before split): 0.9551 +- 0.0089
RIGHT  (scale in pipeline) : 0.9552 +- 0.0090
Gap: -0.01 AUC points


**What just happened.** Here the gap rounds to **less than 0.01 AUC points** — this particular leak, on this particular dataset, did not measurably bite. That is exactly the trap: leaks like this one are invisible until the day they aren't (small datasets, imputation that drags in target-correlated structure, temporal ordering, duplicate rows shared across splits). You cannot know in advance which dataset is the one it bites, and you never have to find out: **put every learning step inside the pipeline and the entire class of bug disappears for free.**

## 3. Prevention

- **Proper Data Splitting:** Ensure that training, validation, and test datasets are mutually exclusive, particularly with time-series data where temporal order matters.
- **Feature Selection:** Exclude features that are derived from or are proxies for the target variable.
- **Cross-Validation:** Use robust cross-validation techniques that prevent leakage (e.g., time-series split for temporal data).

Hard-won habits that make the above automatic:

1. **Everything that learns goes in the pipeline.** Scaling, imputation, encoding, feature selection, oversampling — if it looks at `y` or at the data's statistics, it is re-fit inside each fold by `sklearn.pipeline.Pipeline` (or `imblearn` pipelines for samplers).
2. **Ask the provenance question for every feature:** *"Would this exact value have been available at the moment the prediction is made?"* If the honest answer is no, drop it — no matter how much it improves your score.
3. **Touch the test set once.** Freeze it before any exploration, never let preprocessing or selection see it, and spend it exactly one time, at the end.
4. **Time-series data gets temporal splits.** Random splits let the model train on the future; use `TimeSeriesSplit` or a date-based cut instead.

### Practical Example:

Imagine you are building a model to predict customer churn. Suppose you include a feature such as "customer service calls" that is recorded after a churn event is confirmed. This feature may provide information about the churn outcome that wouldn't be available at the time of prediction, leading to data leakage.

### Summary:

Data leakage undermines the reliability of a model by providing it with information that it shouldn't have during training. Preventing leakage is critical to ensure that the model's performance is indicative of its real-world behavior.

The three experiments above put numbers on it: **15 AUC points** of illusion from one target-correlated feature, **12 accuracy points** from selecting features on the wrong side of the split, and a scaler leak worth **less than 0.01** — the lesson is not that some leaks are safe, but that you never know which one will bite, and leak-free pipelines cost nothing.

> ## reference:
>    - What is Data Leakage? - https://machinelearningmastery.com/data-leakage-machine-learning/
>    - Data Leakage on Credit Card data - https://www.kaggle.com/dansbecker/data-leakage
>    - Data Leakage on Quora Question Pair Dataset - https://www.kaggle.com/sudalairajkumar/simple-leaky-exploration-notebook-quora
>    - Cawley, G. C. & Talbot, N. L. C. (2010). *On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation.* Journal of Machine Learning Research 11 — the canonical study of selection-induced leakage.